In [ ]:
!pip install -q xgboost scikit-learn joblib

In [ ]:
import json
import ast
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving buyers_1300.json to buyers_1300.json
Saving properties_3000.csv to properties_3000.csv
Saving training_pairs_65000.csv to training_pairs_65000.csv


In [ ]:
import os
os.listdir("/content")

['.config',
 'arizona_synthetic_properties_65000.csv',
 'properties_3000.csv',
 'training_pairs_65000.csv',
 'buyers_1300.json',
 'sample_data']

In [ ]:
BUYERS_PATH = "/content/buyers_1300.json"
PROPERTIES_PATH = "/content/properties_3000.csv"
TRAIN_PAIRS_PATH = "/content/training_pairs_65000.csv"

MODEL_PATH = "/content/recommender_xgb.pkl"
BUNDLE_PATH = "/content/recommender_bundle.pkl"

In [ ]:
with open(BUYERS_PATH, "r") as f:
    buyers_raw = json.load(f)

buyers_df = pd.DataFrame(buyers_raw)
properties_df = pd.read_csv(PROPERTIES_PATH)
train_df = pd.read_csv(TRAIN_PAIRS_PATH)

print("buyers_df:", buyers_df.shape)
print("properties_df:", properties_df.shape)
print("train_df:", train_df.shape)

display(buyers_df.head(2))
display(properties_df.head(2))
display(train_df.head(2))

buyers_df: (1300, 11)
properties_df: (3000, 13)
train_df: (65000, 37)


,id,name,email,phone,budget_min,budget_max,desired_locations,timeline,must_have_features,inquiry_text,created_at
0,buyer_0001,Liam Wilson 1,buyer1@example.com,555-851-5746,400000,520000,[Phoenix],immediately,"[family-friendly, near top schools, backyard]",We are looking for a home in Phoenix with fami...,2026-04-19T08:31:27.296058Z
1,buyer_0002,Lucas Jackson 2,buyer2@example.com,555-884-3374,350000,470000,[Gilbert],3-6 months,"[family-friendly, near top schools, backyard]",We are looking for a home in Gilbert with fami...,2026-04-19T08:31:27.296107Z


,id,address_label,location,city,price,bedrooms,bathrooms,sqft,neighborhood,property_type,school_score,commute_score,tags
0,AZP000001,Canyon Edge Home #00001,"Las Sendas, Mesa, AZ",Mesa,452220.0,5,3.0,1701,Las Sendas,Condo,6,6,"['pool', 'guest suite', 'gated community', 'ba..."
1,AZP000002,Skyline Grove Home #00002,"Downtown Tempe, Tempe, AZ",Tempe,453247.0,4,2.5,2666,Downtown Tempe,Condo,10,10,"['smart home', 'security system', 'guest suite..."


,buyer_id,listing_id,buyer_name,budget_min,budget_max,desired_locations,timeline,must_have_features,listing_city,listing_price,...,feature_overlap_count,viewed_count,saved_count,skipped_count,tour_requested_count,avg_view_duration,urgency_score,desired_locations_count,must_have_count,label
0,buyer_0001,AZP000963,Liam Wilson 1,400000,520000,"[""Phoenix""]",immediately,"[""family-friendly"", ""near top schools"", ""backy...",Chandler,516704.0,...,0,0,0,1,0,6.0,3,1,3,0
1,buyer_0001,AZP002597,Liam Wilson 1,400000,520000,"[""Phoenix""]",immediately,"[""family-friendly"", ""near top schools"", ""backy...",Tempe,413430.0,...,0,0,0,1,0,29.0,3,1,3,0


In [ ]:
print("BUYERS COLUMNS:")
print(buyers_df.columns.tolist())

print("\nPROPERTIES COLUMNS:")
print(properties_df.columns.tolist())

print("\nTRAIN COLUMNS:")
print(train_df.columns.tolist())

BUYERS COLUMNS:
['id', 'name', 'email', 'phone', 'budget_min', 'budget_max', 'desired_locations', 'timeline', 'must_have_features', 'inquiry_text', 'created_at']

PROPERTIES COLUMNS:
['id', 'address_label', 'location', 'city', 'price', 'bedrooms', 'bathrooms', 'sqft', 'neighborhood', 'property_type', 'school_score', 'commute_score', 'tags']

TRAIN COLUMNS:
['buyer_id', 'listing_id', 'buyer_name', 'budget_min', 'budget_max', 'desired_locations', 'timeline', 'must_have_features', 'listing_city', 'listing_price', 'listing_tags', 'simulated_event_type', 'simulated_duration_seconds', 'price', 'bedrooms', 'bathrooms', 'sqft', 'school_score', 'commute_score', 'within_budget', 'price_gap_from_mid', 'city_match', 'bedroom_match', 'school_match', 'backyard_match', 'garage_match', 'pool_match', 'feature_overlap_count', 'viewed_count', 'saved_count', 'skipped_count', 'tour_requested_count', 'avg_view_duration', 'urgency_score', 'desired_locations_count', 'must_have_count', 'label']


In [ ]:
def safe_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        return [part.strip() for part in x.split(",") if part.strip()]
    return []

def normalize_text(x):
    return str(x).lower().strip()

def get_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

In [ ]:
if "desired_locations" in buyers_df.columns:
    buyers_df["desired_locations"] = buyers_df["desired_locations"].apply(safe_list)

if "must_have_features" in buyers_df.columns:
    buyers_df["must_have_features"] = buyers_df["must_have_features"].apply(safe_list)

for col in ["desired_locations", "must_have_features"]:
    if col in train_df.columns:
        train_df[col] = train_df[col].apply(safe_list)

for col in properties_df.columns:
    if properties_df[col].dtype == object:
        properties_df[col] = properties_df[col].fillna("").astype(str)

for col in train_df.columns:
    if train_df[col].dtype == object:
        train_df[col] = train_df[col].fillna("")

In [ ]:
PROP_ID_COL = get_first_existing(properties_df, ["id", "property_id", "listing_id"])
PROP_CITY_COL = get_first_existing(properties_df, ["city", "listing_city"])
PROP_PRICE_COL = get_first_existing(properties_df, ["price", "price_usd", "listing_price"])
PROP_BED_COL = get_first_existing(properties_df, ["bedrooms"])
PROP_BATH_COL = get_first_existing(properties_df, ["bathrooms"])
PROP_SQFT_COL = get_first_existing(properties_df, ["sqft", "area_sq_ft"])
PROP_TYPE_COL = get_first_existing(properties_df, ["property_type"])
PROP_TAGS_COL = get_first_existing(properties_df, ["tags", "amenities", "listing_tags"])
PROP_SCHOOL_COL = get_first_existing(properties_df, ["school_score"])
PROP_COMMUTE_COL = get_first_existing(properties_df, ["commute_score"])
PROP_ADDRESS_COL = get_first_existing(properties_df, ["address_label", "property_name", "location"])

print("PROP_ID_COL:", PROP_ID_COL)
print("PROP_CITY_COL:", PROP_CITY_COL)
print("PROP_PRICE_COL:", PROP_PRICE_COL)
print("PROP_TAGS_COL:", PROP_TAGS_COL)

PROP_ID_COL: id
PROP_CITY_COL: city
PROP_PRICE_COL: price
PROP_TAGS_COL: tags


In [ ]:
TARGET_COL = "label"

DROP_COLS = [
    "buyer_id",
    "listing_id",
    "buyer_name",
    "desired_locations",
    "must_have_features",
    "inquiry_text",
    "listing_tags",
    "listing_city",
    "simulated_event_type"
]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS + [TARGET_COL]]

# keep only numeric / boolean-like columns
usable_feature_cols = []
for c in feature_cols:
    if pd.api.types.is_numeric_dtype(train_df[c]) or train_df[c].dropna().isin([0, 1, 0.0, 1.0]).all():
        usable_feature_cols.append(c)

feature_cols = usable_feature_cols

print("Target:", TARGET_COL)
print("Feature count:", len(feature_cols))
print(feature_cols[:50])

Target: label
Feature count: 27
['budget_min', 'budget_max', 'listing_price', 'simulated_duration_seconds', 'price', 'bedrooms', 'bathrooms', 'sqft', 'school_score', 'commute_score', 'within_budget', 'price_gap_from_mid', 'city_match', 'bedroom_match', 'school_match', 'backyard_match', 'garage_match', 'pool_match', 'feature_overlap_count', 'viewed_count', 'saved_count', 'skipped_count', 'tour_requested_count', 'avg_view_duration', 'urgency_score', 'desired_locations_count', 'must_have_count']


In [ ]:
unique_buyers = train_df["buyer_id"].dropna().unique()
rng = np.random.default_rng(42)
rng.shuffle(unique_buyers)

split_idx = int(len(unique_buyers) * 0.8)
train_buyers = set(unique_buyers[:split_idx])
valid_buyers = set(unique_buyers[split_idx:])

train_part = train_df[train_df["buyer_id"].isin(train_buyers)].copy()
valid_part = train_df[train_df["buyer_id"].isin(valid_buyers)].copy()

X_train = train_part[feature_cols].fillna(0)
y_train = train_part[TARGET_COL].astype(int)

X_valid = valid_part[feature_cols].fillna(0)
y_valid = valid_part[TARGET_COL].astype(int)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)
print("train positive rate:", y_train.mean())
print("valid positive rate:", y_valid.mean())

train_part: (52000, 37)
valid_part: (13000, 37)
train positive rate: 0.12051923076923077
valid positive rate: 0.125


In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model training complete")

Model training complete


In [ ]:
valid_probs = model.predict_proba(X_valid)[:, 1]
valid_preds = (valid_probs >= 0.5).astype(int)

print("ROC AUC:", roc_auc_score(y_valid, valid_probs))
print()
print(classification_report(y_valid, valid_preds, zero_division=0))

ROC AUC: 1.0

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     11375
           1       1.00      1.00      1.00      1625

    accuracy                           1.00     13000
   macro avg       1.00      1.00      1.00     13000
weighted avg       1.00      1.00      1.00     13000



In [ ]:
bundle = {
    "model": model,
    "feature_cols": feature_cols,
    "property_columns": {
        "id": PROP_ID_COL,
        "city": PROP_CITY_COL,
        "price": PROP_PRICE_COL,
        "bedrooms": PROP_BED_COL,
        "bathrooms": PROP_BATH_COL,
        "sqft": PROP_SQFT_COL,
        "property_type": PROP_TYPE_COL,
        "tags": PROP_TAGS_COL,
        "school_score": PROP_SCHOOL_COL,
        "commute_score": PROP_COMMUTE_COL,
        "address": PROP_ADDRESS_COL
    }
}

joblib.dump(bundle, BUNDLE_PATH)
print("Saved bundle to:", BUNDLE_PATH)

Saved bundle to: /content/recommender_bundle.pkl


In [ ]:
EVENT_WEIGHTS = {
    "listing_viewed": 0.4,
    "message_clicked": 0.6,
    "listing_saved": 0.8,
    "message_replied": 1.0,
    "tour_requested": 1.0,
    "listing_skipped": -0.5
}

def parse_tags(x):
    if pd.isna(x):
        return []
    text = str(x)
    if ";" in text:
        return [normalize_text(t) for t in text.split(";") if t.strip()]
    if "," in text:
        return [normalize_text(t) for t in text.split(",") if t.strip()]
    return [normalize_text(text)] if text.strip() else []

def aggregate_events(events):
    agg = {}

    for e in events:
        lid = e.get("listing_id")
        if lid is None:
            continue

        if lid not in agg:
            agg[lid] = {
                "viewed_count": 0,
                "saved_count": 0,
                "skipped_count": 0,
                "tour_requested_count": 0,
                "message_clicked_count": 0,
                "message_replied_count": 0,
                "avg_view_duration": 0.0,
                "total_view_duration": 0.0,
                "event_score": 0.0
            }

        etype = str(e.get("event_type", "")).strip()
        duration = float(e.get("duration_seconds", 0) or 0)

        if etype == "listing_viewed":
            agg[lid]["viewed_count"] += 1
            agg[lid]["total_view_duration"] += duration
        elif etype == "listing_saved":
            agg[lid]["saved_count"] += 1
        elif etype == "listing_skipped":
            agg[lid]["skipped_count"] += 1
        elif etype == "tour_requested":
            agg[lid]["tour_requested_count"] += 1
        elif etype == "message_clicked":
            agg[lid]["message_clicked_count"] += 1
        elif etype == "message_replied":
            agg[lid]["message_replied_count"] += 1

        agg[lid]["event_score"] += EVENT_WEIGHTS.get(etype, 0.0)

    for lid, v in agg.items():
        if v["viewed_count"] > 0:
            v["avg_view_duration"] = v["total_view_duration"] / v["viewed_count"]

    return agg

In [ ]:
def timeline_to_urgency_score(timeline):
    t = normalize_text(timeline)
    if "immediately" in t or "asap" in t:
        return 1.0
    if "1 month" in t:
        return 0.8
    if "2-3 months" in t:
        return 0.5
    if "3-6 months" in t:
        return 0.2
    return 0.3

def extract_desired_bedrooms(must_have_features):
    text = " ".join([str(x) for x in must_have_features]).lower()
    import re
    m = re.search(r"(\d+)\s*bed", text)
    return int(m.group(1)) if m else None

def feature_overlap_count(buyer_features, listing_tags):
    bf = set(normalize_text(x) for x in buyer_features)
    lt = set(parse_tags(listing_tags))
    return len(bf.intersection(lt))

def build_inference_row(buyer, prop, events_for_listing, feature_cols, prop_cols):
    row = {c: 0 for c in feature_cols}

    price = float(prop[prop_cols["price"]]) if prop_cols["price"] else 0
    bedrooms = float(prop[prop_cols["bedrooms"]]) if prop_cols["bedrooms"] else 0
    bathrooms = float(prop[prop_cols["bathrooms"]]) if prop_cols["bathrooms"] else 0
    sqft = float(prop[prop_cols["sqft"]]) if prop_cols["sqft"] else 0
    city = str(prop[prop_cols["city"]]) if prop_cols["city"] else ""
    tags = str(prop[prop_cols["tags"]]) if prop_cols["tags"] else ""
    school_score = float(prop[prop_cols["school_score"]]) if prop_cols["school_score"] else 0
    commute_score = float(prop[prop_cols["commute_score"]]) if prop_cols["commute_score"] else 0

    budget_min = float(buyer.get("budget_min", 0))
    budget_max = float(buyer.get("budget_max", 0))
    desired_locations = buyer.get("desired_locations", [])
    must_have_features = buyer.get("must_have_features", [])
    timeline = buyer.get("timeline", "")

    desired_bedrooms = extract_desired_bedrooms(must_have_features)

    derived = {
        "budget_min": budget_min,
        "budget_max": budget_max,
        "listing_price": price,
        "price": price,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft": sqft,
        "school_score": school_score,
        "commute_score": commute_score,
        "within_budget": int(budget_min <= price <= budget_max),
        "city_match": int(city in desired_locations),
        "bedroom_match": int(bedrooms >= desired_bedrooms) if desired_bedrooms is not None else 0,
        "feature_overlap_count": feature_overlap_count(must_have_features, tags),
        "urgency_score": timeline_to_urgency_score(timeline),
        "desired_locations_count": len(desired_locations),
        "must_have_count": len(must_have_features),
        "viewed_count": events_for_listing.get("viewed_count", 0),
        "saved_count": events_for_listing.get("saved_count", 0),
        "skipped_count": events_for_listing.get("skipped_count", 0),
        "tour_requested_count": events_for_listing.get("tour_requested_count", 0),
        "avg_view_duration": events_for_listing.get("avg_view_duration", 0.0),
    }

    mid_budget = (budget_min + budget_max) / 2 if (budget_min + budget_max) > 0 else 0
    derived["price_gap_from_mid"] = abs(price - mid_budget)

    if "school" in normalize_text(" ".join(must_have_features) + " " + buyer.get("inquiry_text", "")):
        derived["school_match"] = int(school_score >= 7)
    else:
        derived["school_match"] = 0

    tags_lower = set(parse_tags(tags))
    derived["backyard_match"] = int("backyard" in tags_lower)
    derived["garage_match"] = int("garage" in tags_lower)
    derived["pool_match"] = int("pool" in tags_lower)

    for c in feature_cols:
        if c in derived:
            row[c] = derived[c]

    return row

In [ ]:
def recommend_top_k_from_input(input_object, properties_df, bundle, top_k=3):
    buyer = input_object["buyer"]
    events = input_object.get("events", [])

    model = bundle["model"]
    feature_cols = bundle["feature_cols"]
    prop_cols = bundle["property_columns"]

    event_agg = aggregate_events(events)

    rows = []
    meta = []

    for _, prop in properties_df.iterrows():
        listing_id = prop[prop_cols["id"]]
        listing_events = event_agg.get(listing_id, {})

        feat_row = build_inference_row(
            buyer=buyer,
            prop=prop,
            events_for_listing=listing_events,
            feature_cols=feature_cols,
            prop_cols=prop_cols
        )

        rows.append(feat_row)

        meta.append({
            "listing_id": listing_id,
            "address_label": prop[prop_cols["address"]] if prop_cols["address"] else listing_id,
            "city": prop[prop_cols["city"]] if prop_cols["city"] else "",
            "price": prop[prop_cols["price"]] if prop_cols["price"] else None,
            "bedrooms": prop[prop_cols["bedrooms"]] if prop_cols["bedrooms"] else None,
            "bathrooms": prop[prop_cols["bathrooms"]] if prop_cols["bathrooms"] else None,
            "sqft": prop[prop_cols["sqft"]] if prop_cols["sqft"] else None,
            "property_type": prop[prop_cols["property_type"]] if prop_cols["property_type"] else "",
            "tags": prop[prop_cols["tags"]] if prop_cols["tags"] else ""
        })

    X_pred = pd.DataFrame(rows)[feature_cols].fillna(0)
    scores = model.predict_proba(X_pred)[:, 1]

    result = pd.DataFrame(meta)
    result["model_score"] = scores

    # hard filter: budget window
    bmin = buyer.get("budget_min", 0)
    bmax = buyer.get("budget_max", 10**9)
    result = result[
        (result["price"] >= bmin * 0.85) &
        (result["price"] <= bmax * 1.15)
    ].copy()

    # soft boost for preferred city
    desired_locations = buyer.get("desired_locations", [])
    result["location_boost"] = result["city"].isin(desired_locations).astype(float) * 0.05
    result["final_score"] = result["model_score"] + result["location_boost"]

    return result.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)

In [ ]:
input_object = {
    "buyer": {
        "id": "buyer_custom_001",
        "name": "Sarah Johnson",
        "budget_min": 350000,
        "budget_max": 450000,
        "desired_locations": ["Tempe", "Mesa"],
        "timeline": "2-3 months",
        "must_have_features": ["3 bedrooms", "good schools", "backyard"],
        "inquiry_text": "We are looking for a family-friendly home with good schools and a backyard."
    },
    "events": [
        {
            "listing_id": str(properties_df.iloc[0][PROP_ID_COL]),
            "event_type": "listing_viewed",
            "duration_seconds": 95,
            "source_channel": "email"
        },
        {
            "listing_id": str(properties_df.iloc[0][PROP_ID_COL]),
            "event_type": "listing_saved",
            "duration_seconds": 40,
            "source_channel": "email"
        },
        {
            "listing_id": str(properties_df.iloc[1][PROP_ID_COL]),
            "event_type": "listing_skipped",
            "duration_seconds": 12,
            "source_channel": "sms"
        }
    ]
}

input_object

{'buyer': {'id': 'buyer_custom_001',
  'name': 'Sarah Johnson',
  'budget_min': 350000,
  'budget_max': 450000,
  'desired_locations': ['Tempe', 'Mesa'],
  'timeline': '2-3 months',
  'must_have_features': ['3 bedrooms', 'good schools', 'backyard'],
  'inquiry_text': 'We are looking for a family-friendly home with good schools and a backyard.'},
 'events': [{'listing_id': 'AZP000001',
   'event_type': 'listing_viewed',
   'duration_seconds': 95,
   'source_channel': 'email'},
  {'listing_id': 'AZP000001',
   'event_type': 'listing_saved',
   'duration_seconds': 40,
   'source_channel': 'email'},
  {'listing_id': 'AZP000002',
   'event_type': 'listing_skipped',
   'duration_seconds': 12,
   'source_channel': 'sms'}]}

In [ ]:
bundle = joblib.load(BUNDLE_PATH)
top3 = recommend_top_k_from_input(input_object, properties_df, bundle, top_k=3)
display(top3)

,listing_id,address_label,city,price,bedrooms,bathrooms,sqft,property_type,tags,model_score,location_boost,final_score
0,AZP000001,Canyon Edge Home #00001,Mesa,452220.0,5,3.0,1701,Condo,"['pool', 'guest suite', 'gated community', 'ba...",0.991103,0.05,1.041103
1,AZP002970,Skyline Point House #02970,Mesa,432954.0,4,3.5,3134,Townhome,"['pool', 'family room', 'mountain views', 'mov...",0.000327,0.05,0.050327
2,AZP000030,Desert Grove House #00030,Tempe,440206.0,4,2.0,3143,Condo,"['near top schools', 'clubhouse', 'move-in rea...",0.000327,0.05,0.050327


In [ ]:
recommended_listings = top3.to_dict(orient="records")
recommended_listings

[{'listing_id': 'AZP000001',
  'address_label': 'Canyon Edge Home #00001',
  'city': 'Mesa',
  'price': 452220.0,
  'bedrooms': 5,
  'bathrooms': 3.0,
  'sqft': 1701,
  'property_type': 'Condo',
  'tags': "['pool', 'guest suite', 'gated community', 'backyard']",
  'model_score': 0.9911030530929565,
  'location_boost': 0.05,
  'final_score': 1.0411030530929566},
 {'listing_id': 'AZP002970',
  'address_label': 'Skyline Point House #02970',
  'city': 'Mesa',
  'price': 432954.0,
  'bedrooms': 4,
  'bathrooms': 3.5,
  'sqft': 3134,
  'property_type': 'Townhome',
  'tags': "['pool', 'family room', 'mountain views', 'move-in ready']",
  'model_score': 0.0003269680601079017,
  'location_boost': 0.05,
  'final_score': 0.050326968060107904},
 {'listing_id': 'AZP000030',
  'address_label': 'Desert Grove House #00030',
  'city': 'Tempe',
  'price': 440206.0,
  'bedrooms': 4,
  'bathrooms': 2.0,
  'sqft': 3143,
  'property_type': 'Condo',
  'tags': "['near top schools', 'clubhouse', 'move-in ready

In [ ]:
def build_outreach_json(input_object, recommended_listings):
    buyer = input_object["buyer"]
    buyer_name = buyer.get("name", "Buyer")

    simplified_recs = []
    for r in recommended_listings[:3]:
        simplified_recs.append({
            "listing_id": r.get("listing_id"),
            "address_label": r.get("address_label"),
            "city": r.get("city"),
            "price": r.get("price"),
            "bedrooms": r.get("bedrooms"),
            "bathrooms": r.get("bathrooms"),
            "property_type": r.get("property_type")
        })

    subject = f"3 home recommendations for {buyer_name}"

    email_lines = [
        f"Hi {buyer_name},",
        "",
        "I found 3 properties that look like strong matches for your requirements.",
        ""
    ]

    for i, r in enumerate(simplified_recs, start=1):
        email_lines.append(
            f"{i}. {r['address_label']} in {r['city']} "
            f"priced at ${int(r['price']):,} "
            f"with {r['bedrooms']} bedrooms and {r['bathrooms']} bathrooms."
        )

    email_lines.extend([
        "",
        "These were selected based on your budget, preferred locations, and feature preferences.",
        "Let me know which one stands out most, and we can set up a tour or share more details.",
        "",
        "Best regards,"
    ])

    email_body = "\n".join(email_lines)

    call_parts = [
        f"Hi {buyer_name}, I found 3 homes that look like strong options for your search."
    ]

    for i, r in enumerate(simplified_recs, start=1):
        ordinal = ["First", "Second", "Third"][i - 1]
        call_parts.append(
            f"The {ordinal} recommendation is {r['address_label']} in {r['city']} "
            f"for ${int(r['price']):,} because it aligns well with your requirements."
        )

    call_parts.append(
        "If one of these stands out, we can set up a tour or I can send you more details."
    )

    call_script = " ".join(call_parts)

    output = {
        "buyer_id": buyer.get("id"),
        "buyer_name": buyer_name,
        "recommended_listings": simplified_recs,
        "email_prompt": {
            "subject": subject,
            "body": email_body
        },
        "call_prompt": {
            "script": call_script
        }
    }

    return output

In [ ]:
outreach_json = build_outreach_json(input_object, recommended_listings)
outreach_json

{'buyer_id': 'buyer_custom_001',
 'buyer_name': 'Sarah Johnson',
 'recommended_listings': [{'listing_id': 'AZP000001',
   'address_label': 'Canyon Edge Home #00001',
   'city': 'Mesa',
   'price': 452220.0,
   'bedrooms': 5,
   'bathrooms': 3.0,
   'property_type': 'Condo'},
  {'listing_id': 'AZP002970',
   'address_label': 'Skyline Point House #02970',
   'city': 'Mesa',
   'price': 432954.0,
   'bedrooms': 4,
   'bathrooms': 3.5,
   'property_type': 'Townhome'},
  {'listing_id': 'AZP000030',
   'address_label': 'Desert Grove House #00030',
   'city': 'Tempe',
   'price': 440206.0,
   'bedrooms': 4,
   'bathrooms': 2.0,
   'property_type': 'Condo'}],
 'email_prompt': {'subject': '3 home recommendations for Sarah Johnson',
  'body': 'Hi Sarah Johnson,\n\nI found 3 properties that look like strong matches for your requirements.\n\n1. Canyon Edge Home #00001 in Mesa priced at $452,220 with 5 bedrooms and 3.0 bathrooms.\n2. Skyline Point House #02970 in Mesa priced at $432,954 with 4 bedr